# Final Report

This notebook will eventually contain the final assignment report for the personalised medicine analysis.

## Planned Report Sections

- Background and aim.
- Datasets and preprocessing.
- Simplified p53 ODE model.
- TCGA-BRCA survival analysis.
- Cell-line doxorubicin response analysis.
- Transfer analysis between cell lines and patients.
- LINCS perturbation signature analysis.
- Discussion, limitations, and conclusion.

## Notes

This report should be written as a clear narrative once the analysis notebooks are complete.

## Linking DepMap/CCLE and PRISM

DepMap/CCLE provides baseline expression measurements for untreated cancer cell lines. PRISM provides measured doxorubicin viability response for many of those same cell lines.

Joining these datasets creates a supervised learning table:

```text
baseline expression -> doxorubicin response
```

This is different from LINCS, which measures drug-induced gene-expression changes after perturbation. In the PRISM table, doxorubicin log-fold-change is a relative cell abundance or viability outcome versus control. It should not be described as the percentage of cells destroyed.

## TCGA-BRCA Survival and Expression Data

TCGA-BRCA provides patient tumour expression data together with clinical follow-up information. This makes it useful for testing whether p53/ODE scores or cell-line-derived expression signatures are associated with patient survival.

Unlike PRISM, TCGA-BRCA does not directly measure doxorubicin response. The TCGA analysis in this assignment is therefore a prognostic clinical analysis, not a direct chemotherapy-response trial.

## Q3 Cell-Line Doxorubicin Signature Transferred to TCGA-BRCA

DepMap/CCLE provides baseline expression for breast cancer cell lines, and PRISM provides measured doxorubicin viability response for the same cell lines. In Q3, a simple elastic-net regression model is used to learn a doxorubicin-sensitivity signature from the shared p53/DNA-damage genes.

The fitted cell-line gene coefficients are then applied to TCGA-BRCA tumour expression to calculate a transferred doxorubicin-sensitivity-like score. This score is tested for association with TCGA overall survival using simple Cox regression and a median-split Kaplan-Meier plot.

This is a prognostic and associational transfer analysis. TCGA-BRCA does not directly measure doxorubicin response, so an association with survival should not be interpreted as proof of chemotherapy benefit.

The apparent cell-line fit and the cross-validated fit answer different questions. The apparent fit shows how well the model explains the 26 training cell lines, while leave-one-out cross-validation gives a more realistic but still unstable estimate of generalisation. The TCGA survival analysis remains the main clinical transfer test and should be interpreted as prognostic/associational rather than direct doxorubicin-response evidence.


## Q4 Patient Survival Signature Applied to PRISM Doxorubicin Response

Q3 asked whether a cell-line drug-response signature was prognostic in patients. Q4 asks the reverse: whether a patient survival signature is related to doxorubicin sensitivity in cell lines.

A TCGA-BRCA Cox model is fitted using only the shared p53/DNA-damage gene-expression variables, because patient covariates such as age and stage cannot be transferred to cell lines. The resulting gene coefficients define a survival-risk score that can be applied to DepMap/CCLE breast cancer cell lines and compared with PRISM doxorubicin sensitivity.

This analysis is exploratory. TCGA overall survival measures patient prognosis, whereas PRISM measures in vitro drug response, so a weak or absent correlation should be reported clearly rather than treated as a failed positive result.


## Formation of the PRISM/DepMap Modelling Cohort

The final PRISM/DepMap breast cancer modelling cohort was formed by joining baseline DepMap/CCLE expression with PRISM doxorubicin response for the same cell lines. The final n=26 is not the total number of breast cancer models in DepMap; it is the intersection of breast lineage, available expression for the selected p53/DNA-damage genes, matched PRISM doxorubicin treatment metadata, and non-missing PRISM doxorubicin log-fold-change response.

This small sample size is an important limitation for Q3 and Q4. It helps explain why the Q3 cross-validation check was unstable and why the PRISM transfer analyses should be interpreted cautiously.


## Static Expression Snapshots and the p53 ODE Model

TCGA-BRCA and DepMap/CCLE provide baseline expression snapshots, not p53 time-course measurements. The ODE model is therefore not learned from TCGA or PRISM time-course data. Instead, the baseline expression profile is used to parameterise or score a pre-specified p53 DNA-damage response model from the course template. The model is then simulated across a standardised DDR input range, producing p53 S15 and p53 S46 response summaries that can be tested against survival or drug response.


## Q1 p53 Model and TCGA-BRCA Survival

Q1 is a prognostic TCGA-BRCA analysis. It tests whether simulated p53 DDR response features, especially p53 S46 AUC across DDR doses, are associated with overall survival. TCGA-BRCA does not directly measure doxorubicin response, so any association should be interpreted as a survival/prognosis signal rather than direct chemotherapy sensitivity.


## Q2 p53 Model and PRISM Doxorubicin Response

Q2 tests the same p53 DDR response features against measured PRISM doxorubicin sensitivity in breast cancer cell lines. PRISM log-fold-change is a viability or relative-abundance outcome, not gene-expression change. More negative PRISM LFC means lower abundance versus control, and the convenience variable `doxorubicin_sensitivity_score` is defined so that higher values mean greater apparent sensitivity. The PRISM/DepMap cohort has only 26 breast cancer cell lines, so weak or negative associations should not be overinterpreted.


## p53 ODE Analysis Limitations

The ODE analyses depend on the biological assumptions and fitted parameters in the course template. TCGA survival is not a direct doxorubicin-response endpoint. PRISM cell-line analysis is small (n=26), and cell lines lack tumour microenvironment, immune context, treatment heterogeneity and patient clinical factors. Negative or weak associations are still informative and should be reported clearly.


## Q5 Model Comparison Synthesis

Q5 compares the mechanistic p53 ODE scores with the regression-based Q3 and Q4 transfer analyses. The p53 ODE model is mechanistic and interpretable because it uses the given course template to simulate p53 DNA-damage response features. The PRISM elastic-net model is data-driven and learns doxorubicin sensitivity from cell-line expression. The TCGA Cox score is patient-derived and prognostic.

| Analysis | Model type | Endpoint | Result | Evidence |
| --- | --- | --- | --- | --- |
| p53 ODE S46 AUC -> TCGA survival | mechanistic ODE | overall survival | HR 0.85, p=0.038; age-adjusted HR 0.84, p=0.027 | modest positive evidence |
| p53 ODE S46 AUC -> PRISM doxorubicin sensitivity | mechanistic ODE | doxorubicin sensitivity | Spearman r=0.04, p=0.838; Pearson r=0.01, p=0.961 | weak or absent evidence |
| PRISM elastic-net signature -> PRISM doxorubicin sensitivity, in-sample | elastic-net regression | doxorubicin sensitivity | R2=0.44, RMSE=1.10, Spearman r=0.76, p=6.66e-06 | apparent fit only |
| PRISM elastic-net signature -> PRISM doxorubicin sensitivity, LOOCV | elastic-net regression | doxorubicin sensitivity | LOOCV R2=-0.83, RMSE=1.99, Spearman r=-0.22, p=0.279 | poor generalisation |
| PRISM elastic-net signature transferred to TCGA survival | elastic-net regression transfer | overall survival | HR=0.95, p=0.603 | weak or absent evidence |
| TCGA Cox gene score -> TCGA survival | Cox regression | overall survival | HR=2.72, p=9.5e-08 | strong within-domain evidence |
| TCGA Cox gene score transferred to PRISM doxorubicin sensitivity | Cox regression transfer | doxorubicin sensitivity | Spearman r=0.01, p=0.975 | weak or absent evidence |

![Q5 model comparison summary](../figures/q5_model_comparison_summary.png)

Overall, the strongest signals were within-domain. The p53 ODE S46 AUC score showed a modest association with TCGA-BRCA survival, and the TCGA-derived Cox gene score was prognostic inside TCGA-BRCA. In contrast, the p53 ODE score did not predict PRISM doxorubicin sensitivity, the PRISM-derived doxorubicin signature did not transfer to TCGA survival, and the TCGA-derived survival score did not transfer to PRISM doxorubicin sensitivity. This supports the conclusion that TCGA survival prognosis and PRISM drug sensitivity are related biological questions but are not interchangeable endpoints.